# **Illustrated Word2Vec**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# 📖 TABLE OF CONTENTS

- [Setup](#Setup)
- [Introduction](#Introduction)
- [Personality Embeddings: What are you like?](#Personality-Embeddings:-What-are-you-like?)
- [References](#References)

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Setup

Before we jump in, let's quickly run the setup cell. It checks the environment (GPU, library versions) and sets random seeds for reproducibility — we want identical results on every run.

In [ ]:
# 🔧 Setup: Run this cell first!
# Environment check + random seeds for reproducibility

import sys
import time
import random
import numpy as np
import pandas as pd
import matplotlib as mpl
import torch
import torch.nn as nn

print(f"📦 Python       {sys.version.split()[0]}")
print(f"🔢 NumPy        {np.__version__}")
print(f"🐼 Pandas       {pd.__version__}")
print(f"📊 Matplotlib   {mpl.__version__}")
print(f"🔥 PyTorch      {torch.__version__}")

# GPU check (only needed for optional deep-feature experiments)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device("cpu")
    print("⚠️ No GPU detected")

DEVICE_LABEL = "GPU" if device.type == "cuda" else "CPU"

# Random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"\n🎲 Random seed set to {SEED} (Python, NumPy, PyTorch)")

%matplotlib inline
%config InlineBackend.figure_format = 'jpeg'   # photos as JPEG, not PNG — ~10× smaller notebook
# Remove character limits for all columns of Pandas dataframes permanently in this session
pd.set_option('display.max_colwidth', None)

📦 Python       3.13.15
🔢 NumPy        2.1.3
🐼 Pandas       2.2.3
📊 Matplotlib   3.10.0
🔥 PyTorch      2.11.0+cpu
⚠️ No GPU detected

🎲 Random seed set to 42 (Python, NumPy, PyTorch)


![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Introduction

The concept of embeddings is one of the most fascinating ideas in machine learning. If you’ve ever used Siri, Google Assistant, Alexa, Google Translate, or even smartphone keyboard with next-word prediction, then chances are you’ve benefitted from this idea that has become central to Natural Language Processing (NLP) models. There has been quite a development over the last couple of decades in using embeddings for neural models.

Word2vec is a method to efficiently create word embeddings and has been around since 2013. But in addition to its utility as a word-embedding method, some of its concepts have been shown to be effective in creating recommendation engines and making sense of sequential data even in commercial, non-language tasks. Companies like Airbnb, Alibaba, Spotify, and Anghami have all benefitted from carving out this brilliant piece of machinery from the world of NLP and using it in production to empower a new breed of recommendation engines.

In this notebook, we’ll go over the concept of embedding, and the mechanics of generating embeddings with Word2Vec. But let’s start with an example to get familiar with using vectors to represent things.

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Personality Embeddings: What are you like?

On a scale of 0 to 100, how introverted/extroverted are you (where 0 is the most introverted, and 100 is the most extroverted)? Have you ever taken a personality test like MBTI – or even better, the Big Five Personality Traits test? If you haven’t, these are tests that ask you a list of questions, then score you on a number of axes, introversion/extroversion being one of them.

<table>
  <tr>
    <td align="center">
      <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-01.png" width="400">
    </td>
  </tr>
  <tr>
    <td align="center">
      <em>Example of the result of a Big Five Personality Trait test. It can really tell you a lot about yourself and is shown to have predictive ability in academic, personal, and professional success.</em>
    </td>
  </tr>
</table>

Imagine I’ve scored 38/100 as my introversion/extroversion score. we can plot that in this way:

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-02.png" width="400">

Let’s switch the range to be from -1 to 1:

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-03.png" width="400">

How well do you feel you know a person knowing only this one piece of information about them? Not much. People are complex. So let’s add another dimension – the score of one other trait from the test.

<table>
  <tr>
    <td align="center">
      <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-04.png" width="400">
    </td>
  </tr>
  <tr>
    <td align="center">
      <em>We can represent the two dimensions as a point on the graph, or better yet, as a vector from the origin to that point. We have incredible tools to deal with vectors that will come in handy very shortly.</em>
    </td>
  </tr>
</table>

I’ve hidden which traits we’re plotting just so you get used to not knowing what each dimension represents – but still getting a lot of value from the vector representation of a person’s personality.

We can now say that this vector partially represents my personality. The usefulness of such representation comes when you want to compare two other people to me. Say I get hit by a $\color{red}{\text{bus}}$ and I need to be replaced by someone with a similar personality. In the following figure, which of the two people is more similar to me?

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-05.png" width="500">

When dealing with vectors, a common way to calculate a similarity score is cosine similarity:

<table>
  <tr>
    <td align="center">
      <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-06.png" width="500">
    </td>
  </tr>
  <tr>
    <td align="center">
      <em>Person #1 is more similar to me in personality. Vectors pointing at the same direction (length plays a role as well) have a higher cosine similarity score.</em>
    </td>
  </tr>
</table>

Yet again, two dimensions aren’t enough to capture enough information about how different people are. Decades of psychology research have led to five major traits (and plenty of sub-traits). So let’s use all five dimensions in our comparison:

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-07.png" width="400">

The problem with five dimensions is that we lose the ability to draw neat little arrows in two dimensions. This is a common challenge in machine learning where we often have to think in higher-dimensional space. The good thing is, though, that cosine_similarity still works. It works with any number of dimensions:

<table>
  <tr>
    <td align="center">
      <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-08.png" width="500">
    </td>
  </tr>
  <tr>
    <td align="center">
      <em>cosine_similarity works for any number of dimensions. These are much better scores because they're calculated based on a higher resolution representation of the things being compared.</em>
    </td>
  </tr>
</table>

At the end of this section, I want us to come out with two central ideas:

1. We can represent people (and things) as vectors of numbers (which is great for machines!).
2. We can easily calculate how similar vectors are to each other.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-09.png" width="500">

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# References

- **[The Illustrated Word2vec by Jay Alammar](https://jalammar.github.io/illustrated-word2vec/)**
- **Key Papers:**
    - [Word2Vec Paper, 2013](https://arxiv.org/pdf/1301.3781)

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-01.png" width="400">

<table>
  <tr>
    <td align="center">
      <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-01.png" width="400">
    </td>
  </tr>
  <tr>
    <td align="center">
      <em>Figure 1: Word Embeddings</em>
    </td>
  </tr>
</table>

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)